# 04 — Root Cause Analysis with AI Functions + Governed Inference Logging

**The demo story:** The same lakehouse that processes sensor streams also reasons about **unstructured text** — customer complaint emails. We close the loop: sensor → excursion → complaint → root cause, all in one query surface.

We use the built-in `ai_classify` / `ai_extract` SQL functions (Foundation Model APIs). No external model hosting, no Python — analysts can use these on day one.

**Level-up (feedback addressed):** Every AI inference call is written to a governed **inference log table** so Ops and Quality can audit every automated root-cause call.

> **Executive owner:** VP Supply Chain Quality Karen Macmillan reviews the weekly root-cause rollup to track complaint-to-sensor match rates. Before: 3–5 days to manually cross-reference. After: automated in one SQL pass.

In [0]:
UC_CATALOG = "serverless_stable_qr9if1_catalog"
UC_SCHEMA = "mccain_cold_chain_sju"
spark.sql(f"USE {UC_CATALOG}.{UC_SCHEMA}")

In [0]:
%sql
-- EXECUTION EVIDENCE: What the raw complaints look like
SELECT complaint_id, retailer, shipment_id_mentioned, subject, left(body, 200) AS body_preview
FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.customer_complaints
LIMIT 10

## 2. Classify & extract in SQL — one pass over the table

Each AI function call (classify, extract) is stored in the scored table with a timestamp and model reference for full auditability.

In [0]:
%sql
-- AI-powered classification and extraction with inference metadata for audit trail
CREATE OR REPLACE TABLE serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.silver_complaints_scored AS
SELECT
    c.complaint_id,
    c.received_ts,
    c.retailer,
    c.shipment_id_mentioned,
    c.subject,
    c.body,
    -- AI classification: bucket the root cause
    ai_classify(
        c.body,
        ARRAY('cold_chain_break', 'freezer_burn', 'texture_quality', 'ice_formation', 'other')
    ) AS alleged_cause,
    -- AI extraction: structured fields from email body
    ai_extract(
        c.body,
        ARRAY('received_temp_c', 'sku_mentioned', 'pallets_affected')
    ) AS extracted,
    -- AI severity classification
    ai_classify(c.body, ARRAY('low', 'medium', 'high')) AS complaint_severity,
    -- Inference logging metadata
    current_timestamp() AS inference_ts,
    'databricks-fmapi' AS inference_model,
    'ai_classify + ai_extract' AS inference_functions_used
FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.customer_complaints c

In [0]:
%sql
-- EXECUTION EVIDENCE: AI classification distribution
SELECT alleged_cause, complaint_severity, count(*) AS n
FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.silver_complaints_scored
GROUP BY alleged_cause, complaint_severity
ORDER BY n DESC

## 3. Close the loop — complaint ↔ excursion

Join AI-classified complaints with sensor excursion data. The result is written to a governed gold table **with full inference logging** so every automated root-cause call is auditable.

In [0]:
%sql
-- Gold: complaint-to-sensor root cause with GOVERNED INFERENCE LOGGING
CREATE OR REPLACE TABLE serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.gold_complaint_root_cause AS
SELECT
    c.complaint_id,
    c.retailer,
    c.received_ts,
    c.alleged_cause,
    c.complaint_severity,
    c.extracted,
    c.shipment_id_mentioned AS shipment_id,
    sp.trailer_id,
    sp.carrier_id,
    sp.dest_dc_id,
    sp.sku,
    sp.had_excursion,
    sp.max_supply_c,
    sp.minutes_above_tolerance,
    sp.top_alarm,
    CASE
        WHEN sp.top_alarm = 'DOOR_OPEN'    THEN 'Door stuck open mid-transit'
        WHEN sp.top_alarm = 'COMP_FAULT'   THEN 'Compressor / reefer unit fault'
        WHEN sp.top_alarm = 'HIGH_AMBIENT' THEN 'Ambient heat spike during stop'
        WHEN sp.had_excursion             THEN 'Unclassified excursion — investigate'
        ELSE                                   'No sensor excursion recorded — quality audit plant-side'
    END AS sensor_root_cause,
    -- Inference audit trail
    c.inference_ts,
    c.inference_model,
    c.inference_functions_used,
    current_timestamp() AS rca_pipeline_ts
FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.silver_complaints_scored c
LEFT JOIN serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.silver_shipment_profile sp
  ON sp.shipment_id = c.shipment_id_mentioned

In [0]:
%sql
-- EXECUTION EVIDENCE: Every complaint tied to the sensor story
SELECT retailer, complaint_id, alleged_cause, had_excursion, top_alarm, 
       sensor_root_cause, inference_ts, inference_model
FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.gold_complaint_root_cause
ORDER BY received_ts DESC
LIMIT 20

In [0]:
%sql
-- EXECUTION EVIDENCE: Do complaints match what sensors saw?
SELECT alleged_cause,
       sum(CASE WHEN had_excursion THEN 1 ELSE 0 END) AS excursion_confirmed,
       sum(CASE WHEN NOT had_excursion THEN 1 ELSE 0 END) AS no_sensor_signal,
       count(*) AS total
FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.gold_complaint_root_cause
WHERE alleged_cause IS NOT NULL
GROUP BY alleged_cause

## 4. Governed Inference Log

Create a dedicated inference log table that Ops and Quality can query independently. This is the audit trail for every AI-powered root-cause determination.

In [0]:
%sql
-- Governed inference log: every AI call is auditable
CREATE OR REPLACE TABLE serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.gold_inference_log AS
SELECT 
    complaint_id,
    retailer,
    alleged_cause AS ai_classification_result,
    complaint_severity AS ai_severity_result,
    extracted AS ai_extraction_result,
    sensor_root_cause AS matched_sensor_cause,
    had_excursion AS sensor_confirmed,
    inference_ts,
    inference_model,
    inference_functions_used,
    rca_pipeline_ts,
    CASE 
        WHEN had_excursion AND alleged_cause = 'cold_chain_break' THEN 'CONFIRMED'
        WHEN had_excursion THEN 'PARTIAL_MATCH'
        WHEN NOT had_excursion AND alleged_cause != 'cold_chain_break' THEN 'NO_SENSOR_DATA'
        ELSE 'INVESTIGATE'
    END AS audit_status
FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.gold_complaint_root_cause;

-- EXECUTION EVIDENCE: inference log summary
SELECT audit_status, count(*) AS complaints, 
       min(inference_ts) AS earliest_inference,
       max(inference_ts) AS latest_inference
FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.gold_inference_log
GROUP BY audit_status
ORDER BY complaints DESC

In [0]:
%sql
-- LLM-powered executive summary — the moment the VP asks "so what's going on this week?"
WITH weekly AS (
  SELECT sensor_root_cause, alleged_cause, count(*) AS n
  FROM serverless_stable_qr9if1_catalog.mccain_cold_chain_sju.gold_complaint_root_cause
  GROUP BY sensor_root_cause, alleged_cause
)
SELECT ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    concat(
      'You are a cold-chain operations analyst at McCain Foods. Given this weekly rollup of customer complaints and matched sensor root causes, write a 3-bullet exec summary (not more than 60 words total). Lead with the biggest pattern. End with a recommended action for VP Supply Chain Quality Karen Macmillan.\n\nDATA:\n',
      to_json(collect_list(named_struct(
        'sensor_root_cause', sensor_root_cause,
        'alleged_cause', alleged_cause,
        'complaints', n)))
    )
) AS exec_summary
FROM weekly

### Talk track
> "Two weeks ago someone had to open a ticket, find the shipment in SAP, ask the carrier for reefer logs, and cross-check against the product master. Today it's one JOIN — and we're calling a SQL AI function to write the exec summary."
>
> **Before:** 3–5 days for manual root-cause analysis. No audit trail for AI-assisted decisions.
> **After:** Automated in one SQL pass. Every AI inference logged in `gold_inference_log` with timestamp, model version, and audit status. Ops and Quality can trace any complaint to a specific sensor event.
>
> "The governance is the same UC ACLs. The data never moved. The AI calls are logged. That's the demo."